In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer

from estrus.data import KEYS, load_labels
from estrus.features import assemble_features
from estrus.modeling import evaluate, split_mice

ROOT = Path.cwd().resolve().parent
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path.cwd().resolve()
CONFIG_PATH = ROOT / "Model Building" / "config.json"
CONFIG = json.loads(CONFIG_PATH.read_text())
print(f"Repository: {ROOT}")
print(f"Configured model: {CONFIG_PATH}")


Repository: C:\Users\bvhaw\capstone-estrus-classification
Configured model: C:\Users\bvhaw\capstone-estrus-classification\Model Building\config.json


In [2]:
WORKSPACE = ROOT / "data" / "outputs" / "notebook_features"
features, diagnostics = assemble_features(ROOT, WORKSPACE, baseline_only=False)
labels = load_labels()
feature_columns = [column for column in features.columns if column not in KEYS]

print(f"Rows: {len(features)}")
print(f"Features: {len(feature_columns)}")
print("Accepted feature exports:")
for item in diagnostics:
    print(f"  {item['owner']}: {item['status']}" + (f" ({item.get('n_features', 0)} features)" if item['status'] == 'accepted' else ""))
features.head()


[Matthew: accepted] 4 features
[Vivian: accepted] 6 features
[Brandon: missing] No export yet; skipped.
[Alisha: accepted] 18 features
Feature inputs: 28 contributor features only; reference features excluded.
Rows: 156
Features: 28
Accepted feature exports:
  Matthew: accepted (4 features)
  Vivian: accepted (6 features)
  Brandon: missing
  Alisha: accepted (18 features)


,mouse_id,day_index,matthew__circadian_la_amplitude_next_day_delta,matthew__cbt_ultradian_amplitude_h0_12,matthew__cbt_fraction_above_37_2_h6_12,matthew__la_fraction_above_15_h6_12,vivian__adj_ultradian_amp,vivian__adj_circadian_amp,vivian__adj_mean_h6_12,vivian__adj_range,...,alisha__dtw_shift_seg8_temp,alisha__dtw_shift_whole_act,alisha__dtw_shift_seg1_act,alisha__dtw_shift_seg2_act,alisha__dtw_shift_seg3_act,alisha__dtw_shift_seg4_act,alisha__dtw_shift_seg5_act,alisha__dtw_shift_seg6_act,alisha__dtw_shift_seg7_act,alisha__dtw_shift_seg8_act
0,f1,2,-1.339682,0.234494,0.775000,0.491667,0.491294,2.922951,37.654729,2.019403,...,6.637399,-5.923617,-1.297728,-7.375809,-5.141982,15.526077,4.594079,1.613962,-9.883207,-2.198737
1,f1,3,0.061230,0.381929,0.727778,0.513889,0.527014,2.908079,37.359077,1.926110,...,13.864996,0.360182,-2.277469,12.374668,-11.975557,-4.210786,-6.880210,-9.565535,-5.654289,1.460430
2,f1,4,1.194861,0.266938,0.636111,0.475000,0.414884,2.736556,37.416864,2.001510,...,-8.272010,-13.617739,5.882488,-2.515704,0.308997,-18.379950,-30.966134,11.047435,3.887959,-0.005333
3,f1,5,2.140203,0.250355,0.630556,0.469444,0.602273,2.285139,37.403446,2.328635,...,2.498478,20.289367,-5.162133,1.320633,16.449685,-6.166616,15.286544,-0.798174,0.045145,4.156510
4,f1,6,-1.069772,0.183187,1.000000,0.625000,0.540862,2.072434,37.638715,2.203336,...,9.362160,-1.096327,2.832335,-1.614599,-3.943827,2.906270,28.112993,-2.880232,-15.265771,-10.562849


In [3]:
# Calculate MI on development mice only, keeping the held-out mice untouched.
frame = features.merge(labels.dropna(subset=["is_estrus"]), on=KEYS, how="inner", validate="one_to_one")
y = frame["is_estrus"].astype(int).to_numpy()
groups = frame["mouse_id"].to_numpy()
train_indices, test_indices = split_mice(y, groups, CONFIG)

X_train = frame.iloc[train_indices][feature_columns]
y_train = y[train_indices]
imputed = SimpleImputer(strategy="median", keep_empty_features=True).fit_transform(X_train)
mi_values = mutual_info_regression(imputed, y_train, random_state=CONFIG["seed"])
mi_scores = (
    pd.DataFrame({"feature": feature_columns, "mi_score": mi_values})
    .sort_values("mi_score", ascending=False)
    .reset_index(drop=True)
)
mi_scores


,feature,mi_score
0,alisha__dtw_shift_whole_temp,0.488053
1,matthew__cbt_fraction_above_37_2_h6_12,0.342729
2,matthew__cbt_ultradian_amplitude_h0_12,0.306085
3,matthew__la_fraction_above_15_h6_12,0.195091
4,vivian__adj_range,0.187722
5,matthew__circadian_la_amplitude_next_day_delta,0.144206
6,alisha__dtw_shift_seg4_temp,0.136030
7,alisha__dtw_shift_seg1_temp,0.130624
8,vivian__adj_mean_h6_12,0.121244
9,alisha__dtw_shift_seg3_temp,0.099528


In [4]:
run_id = datetime.now(timezone.utc).strftime("notebook-run-%Y%m%dT%H%M%S%fZ")
model_output = ROOT / "Model Building" / "results" / run_id
predictions, metrics = evaluate(features, labels, model_output, CONFIG, root=ROOT)

print(f"Model output: {model_output}")
metrics


Compare feature sets and models on 10 training mice; reserve 3 test mice: f11, f3, f6


Chosen: logistic, Top 35% of features. Fitting one model with probability calibration...


Assessing 28 features using training mice only...


Model output: C:\Users\bvhaw\capstone-estrus-classification\Model Building\results\notebook-run-20261113T231216701940Z


,metric,estimate
0,average_precision,0.932099
1,balanced_accuracy,0.777778
2,precision,1.000000
3,recall,0.555556
4,brier,0.068100
5,log_loss,0.234408
6,brier_improvement_over_prevalence,0.119400


In [5]:
feature_summary = pd.read_csv(model_output / "feature_summary.csv")
relevance = mi_scores.merge(feature_summary[["feature", "individual_ap", "importance_ap_drop", "used"]], on="feature")
relevance


,feature,mi_score,individual_ap,importance_ap_drop,used
0,alisha__dtw_shift_whole_temp,0.488053,0.977513,0.054733,True
1,matthew__cbt_fraction_above_37_2_h6_12,0.342729,0.928517,0.017382,True
2,matthew__cbt_ultradian_amplitude_h0_12,0.306085,0.866703,-0.000464,True
3,matthew__la_fraction_above_15_h6_12,0.195091,0.762682,-0.000900,True
4,vivian__adj_range,0.187722,0.619117,0.033811,True
5,matthew__circadian_la_amplitude_next_day_delta,0.144206,0.737601,0.000965,True
6,alisha__dtw_shift_seg4_temp,0.136030,0.674525,0.000000,True
7,alisha__dtw_shift_seg1_temp,0.130624,0.385069,0.000000,False
8,vivian__adj_mean_h6_12,0.121244,0.636835,0.001621,False
9,alisha__dtw_shift_seg3_temp,0.099528,0.564167,0.004198,True
